In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==========================================================
# NOTEBOOK 3
# Ensemble Learning
# - Simple Average (SA)
# - Inverse-MSE Weighted Average (IMSE)
# ==========================================================

import os
import warnings
import numpy as np
import pandas as pd

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

warnings.filterwarnings("ignore")

# ==========================================================
# DIRECTORIES
# ==========================================================

TABLE_DIR = "/content/drive/MyDrive/Biomass_Project/10_Tables/Nested_CV"

OUTPUT_DIR = os.path.join(TABLE_DIR, "Ensemble")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================================
# BASE MODELS
# ==========================================================

MODELS = [
    "CatBoost",
    "RF",
    "XGB",
    "LGBM",
    "SVR"
]

print("Output directory:")
print(OUTPUT_DIR)

Output directory:
/content/drive/MyDrive/Biomass_Project/10_Tables/Nested_CV/Ensemble


In [ ]:
# ==========================================================
# LOAD PREDICTION FILES
# ==========================================================

prediction_data = {}

for model in MODELS:

    file_path = os.path.join(TABLE_DIR, f"{model}_Predictions.csv")

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"{file_path} not found.")

    prediction_data[model] = pd.read_csv(file_path)

    print(f"{model:<10} : {prediction_data[model].shape}")

CatBoost   : (1227, 4)
RF         : (1227, 4)
XGB        : (1227, 4)
LGBM       : (1227, 4)
SVR        : (1227, 4)


In [ ]:
# ==========================================================
# VERIFY ALIGNMENT
# ==========================================================

reference = prediction_data[MODELS[0]]

for model in MODELS[1:]:

    df = prediction_data[model]

    assert reference["Observed"].equals(df["Observed"]), \
        f"Observed values do not match: {model}"

    assert reference["Fold"].equals(df["Fold"]), \
        f"Fold numbers do not match: {model}"

    assert reference["Index"].equals(df["Index"]), \
        f"Indices do not match: {model}"

print("✓ All prediction files are perfectly aligned.")

✓ All prediction files are perfectly aligned.


In [ ]:
# ==========================================================
# CREATE PREDICTION MATRIX
# ==========================================================

y_true = reference["Observed"].values

pred_matrix = np.column_stack([
    prediction_data[model]["Predicted"].values
    for model in MODELS
])

print("Prediction matrix shape:", pred_matrix.shape)

Prediction matrix shape: (1227, 5)


In [ ]:
# ==========================================================
# MODEL EVALUATION FUNCTION
# ==========================================================

def evaluate_model(y_true, y_pred):

    r2 = r2_score(y_true, y_pred)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    mae = mean_absolute_error(y_true, y_pred)

    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    rrmse = (rmse / np.mean(y_true)) * 100

    return {
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "rRMSE": rrmse
    }

In [ ]:
# ==========================================================
# SIMPLE AVERAGE (SA)
# ==========================================================

# Average predictions from all base models
sa_pred = pred_matrix.mean(axis=1)

# Evaluate
sa_results = evaluate_model(y_true, sa_pred)

print("\nSimple Average Performance")
print("-" * 40)

for metric, value in sa_results.items():
    print(f"{metric:<8}: {value:.4f}")


Simple Average Performance
----------------------------------------
R2      : 0.7407
RMSE    : 32.0114
MAE     : 19.4529
MAPE    : 18.4003
rRMSE   : 27.3141


In [ ]:
# ==========================================================
# SAVE SA PREDICTIONS
# ==========================================================

sa_df = reference[["Fold", "Index", "Observed"]].copy()

sa_df["Predicted"] = sa_pred

sa_path = os.path.join(OUTPUT_DIR, "SA_predictions.csv")

sa_df.to_csv(sa_path, index=False)

print(f"\nSaved: {sa_path}")


Saved: /content/drive/MyDrive/Biomass_Project/10_Tables/Nested_CV/Ensemble/SA_predictions.csv


In [ ]:
# ==========================================================
# LOAD FOLD METRICS
# ==========================================================

fold_metrics = {}

for model in MODELS:

    file_path = os.path.join(TABLE_DIR, f"{model}_Fold_Metrics.csv")

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"{file_path} not found.")

    fold_metrics[model] = pd.read_csv(file_path)

print("✓ Fold metrics loaded.")


# ==========================================================
# LEAKAGE-FREE IMSE INITIALIZATION
# ==========================================================

imse_pred = np.zeros(len(y_true))

weight_history = []

✓ Fold metrics loaded.


In [ ]:
# ==========================================================
# LEAKAGE-FREE IMSE
# ==========================================================

unique_folds = sorted(reference["Fold"].unique())

for fold in unique_folds:

    # ---------------------------------------------
    # Compute weights using ONLY the remaining folds
    # ---------------------------------------------

    inverse_mse = {}

    for model in MODELS:

        df = fold_metrics[model]

        train_rmse = df.loc[df["Fold"] != fold, "RMSE"]

        mean_mse = np.mean(train_rmse ** 2)

        inverse_mse[model] = 1 / mean_mse

    total_inverse = sum(inverse_mse.values())

    weights = {
        model: inverse_mse[model] / total_inverse
        for model in MODELS
    }

    # Save weights for later inspection
    weight_history.append(
        {
            "Fold": fold,
            **weights
        }
    )

    # ---------------------------------------------
    # Predict ONLY current fold
    # ---------------------------------------------

    fold_mask = reference["Fold"] == fold

    fold_predictions = pred_matrix[fold_mask]

    weight_vector = np.array([
        weights[model]
        for model in MODELS
    ])

    imse_pred[fold_mask] = fold_predictions @ weight_vector

print("✓ Leakage-free IMSE completed.")

✓ Leakage-free IMSE completed.


In [ ]:
# ==========================================================
# EVALUATE IMSE
# ==========================================================

imse_results = evaluate_model(y_true, imse_pred)

print("\nLeakage-Free IMSE Performance")
print("-" * 40)

for metric, value in imse_results.items():
    print(f"{metric:<8}: {value:.4f}")


Leakage-Free IMSE Performance
----------------------------------------
R2      : 0.7405
RMSE    : 32.0183
MAE     : 19.4650
MAPE    : 18.4066
rRMSE   : 27.3200


In [ ]:
# ==========================================================
# SAVE IMSE WEIGHTS
# ==========================================================

weights_df = pd.DataFrame(weight_history)

weights_path = os.path.join(
    OUTPUT_DIR,
    "IMSE_Fold_Weights.csv"
)

weights_df.to_csv(weights_path, index=False)

print(weights_df.round(4))

   Fold  CatBoost      RF     XGB    LGBM     SVR
0     1    0.2082  0.2046  0.2013  0.2009  0.1850
1     2    0.2092  0.2034  0.2038  0.2019  0.1816
2     3    0.2058  0.2048  0.2047  0.1998  0.1849
3     4    0.2062  0.2067  0.2045  0.2002  0.1825
4     5    0.2069  0.2042  0.1953  0.1995  0.1941


In [ ]:
# ==========================================================
# SAVE IMSE PREDICTIONS
# ==========================================================

imse_df = reference[["Fold", "Index", "Observed"]].copy()

imse_df["Predicted"] = imse_pred

imse_path = os.path.join(
    OUTPUT_DIR,
    "IMSE_predictions.csv"
)

imse_df.to_csv(imse_path, index=False)

print(f"Saved: {imse_path}")

Saved: /content/drive/MyDrive/Biomass_Project/10_Tables/Nested_CV/Ensemble/IMSE_predictions.csv


In [ ]:
# ==========================================================
# PERFORMANCE SUMMARY
# ==========================================================

performance_df = pd.DataFrame([
    {
        "Model": "SA",
        **sa_results
    },
    {
        "Model": "IMSE",
        **imse_results
    }
])

performance_df = performance_df.round(4)

performance_path = os.path.join(
    OUTPUT_DIR,
    "Ensemble_Performance.csv"
)

performance_df.to_csv(
    performance_path,
    index=False
)

print(performance_df)

  Model      R2     RMSE      MAE     MAPE    rRMSE
0    SA  0.7407  32.0114  19.4529  18.4003  27.3141
1  IMSE  0.7405  32.0183  19.4650  18.4066  27.3200


In [ ]:
# ==========================================================
# SUMMARY
# ==========================================================

print("=" * 60)
print("ENSEMBLE LEARNING COMPLETED")
print("=" * 60)

print("\nModels:")
print(", ".join(MODELS))

print("\nPerformance")
print(performance_df)

print("\nSaved Files")
print(f"• {sa_path}")
print(f"• {imse_path}")
print(f"• {weights_path}")
print(f"• {performance_path}")

print("=" * 60)

ENSEMBLE LEARNING COMPLETED

Models:
CatBoost, RF, XGB, LGBM, SVR

Performance
  Model      R2     RMSE      MAE     MAPE    rRMSE
0    SA  0.7407  32.0114  19.4529  18.4003  27.3141
1  IMSE  0.7405  32.0183  19.4650  18.4066  27.3200

Saved Files
• /content/drive/MyDrive/Biomass_Project/10_Tables/Nested_CV/Ensemble/SA_predictions.csv
• /content/drive/MyDrive/Biomass_Project/10_Tables/Nested_CV/Ensemble/IMSE_predictions.csv
• /content/drive/MyDrive/Biomass_Project/10_Tables/Nested_CV/Ensemble/IMSE_Fold_Weights.csv
• /content/drive/MyDrive/Biomass_Project/10_Tables/Nested_CV/Ensemble/Ensemble_Performance.csv
